In [1]:
from matplotlib import pyplot as plt
import torch
from sklearn.metrics import roc_curve, f1_score, auc
from tools.embs_tools import get_embs, accelerated_cosine_similarity, get_crops_for_id
import numpy as np
import cv2

In [2]:
all_embs = get_embs("/home/simon/new_outputs/")
print(all_embs.keys())

ERROR:root:No meta file found in /home/simon/new_outputs/SD-PO-P576__20250519-062004/2025_07/12/19/24/track_meta.csv, skipping...


dict_keys(['/home/simon/new_outputs/PJ-CE-P551__20250519-050004/2025_07/12/16/21', '/home/simon/new_outputs/SD-PO-P576__20250519-054004/2025_07/12/18/53', '/home/simon/new_outputs/JS-BM-P4951__20250519-042004/2025_07/12/07/47', '/home/simon/new_outputs/SG-CE-P574__20250519-060004/2025_07/12/21/04', '/home/simon/new_outputs/JS-BM-P490__20250519-054005/2025_07/12/01/50', '/home/simon/new_outputs/JS-BM-P498__20250519-042013/2025_07/12/09/56', '/home/simon/new_outputs/JS-BM-P489__20250519-042011/2025_07/11/21/52', '/home/simon/new_outputs/JS-BM-P491__20250519-062005/2025_07/12/04/57', '/home/simon/new_outputs/JS-BM-P489__20250519-060005/2025_07/11/23/36', '/home/simon/new_outputs/JS-BM-P491__20250519-042011/2025_07/12/03/01', '/home/simon/new_outputs/JS-BM-P490__20250519-044005/2025_07/12/00/49', '/home/simon/new_outputs/JS-BM-P488__20250519-054013/2025_07/11/20/25', '/home/simon/new_outputs/JK-CE-P571__20250519-054004/2025_07/11/15/01', '/home/simon/new_outputs/JS-BM-P4951__20250519-05200

In [3]:
accs = 0
fprs = 0
tprs = 0

for path in list(all_embs.keys()):
    thr = 0.466

    avg_fn = lambda embs: np.mean(embs, axis=0)
    embs = torch.stack([torch.tensor(emb) for _, _, emb in all_embs[path]])
    ids = torch.tensor([id for id, _, _ in all_embs[path]])

    embs = embs[:50000]
    ids = ids[:50000]

    similarity = accelerated_cosine_similarity(embs, embs, batch_size=2048)
    similarity_flat = similarity.flatten()
    issame_tracks = (ids.unsqueeze(0) == ids.unsqueeze(1))
    issame_tracks_flat = issame_tracks.flatten()

    def compute_top1_metrics(similarity: torch.Tensor, issame_tracks: torch.Tensor):
        similarity_no_diag = similarity.clone()
        diag_indices = torch.arange(similarity.size(0))
        similarity_no_diag[diag_indices, diag_indices] = -float('inf')

        # Find top 1 match indices for each row
        top1_indices = torch.argmax(similarity_no_diag, dim=1)
        top1_similarities, _ = torch.max(similarity_no_diag, dim=1)

        # Calculate True Positives (TP): top1 identified sample is indeed the same track
        tp = (issame_tracks[diag_indices, top1_indices] == 1).sum().item()

        # Calculate False Positives (FP): top1 identified sample is not the same track
        fp = (issame_tracks[diag_indices, top1_indices] == 0).sum().item()

        # Calculate total positives and negatives excluding diagonal
        total_positive = issame_tracks.sum().item() - similarity.size(0)  # exclude diagonal
        total_negative = issame_tracks.numel() - issame_tracks.sum().item() - similarity.size(0)

        # Compute rates and accuracy
        tpir = tp / total_positive if total_positive > 0 else 0.0
        fpir = fp / total_negative if total_negative > 0 else 0.0
        accuracy = tp / similarity.size(0)

        return tpir, fpir, accuracy

    tpir, fpir, acc = compute_top1_metrics(similarity, issame_tracks)

    accs += acc
    fprs += fpir
    tprs += tpir

print(accs / len(all_embs))
print(tprs / len(all_embs))
print(fprs / len(all_embs))


ValueError: too many values to unpack (expected 3)